# Colab 07 — V3.1 Missed-Move Diagnostics + Frozen Development Test

**OFFLINE ONLY.** Analyze an existing private MICRO ZIP; never mount Google Drive here. No real source reads, source writes, Validation 2025 or Final OOS 2026. This notebook does not create owner approval. Use Colab 04 in a **separate owner session** only for each explicitly authorized *new Development* month.

- **2024-07:** repeatedly explored Development, descriptive diagnostics only.
- **2024-08 → 2024-09 → 2024-10:** frozen chronological Development sequence. Start with one month at a time, explicit approval per new source read. No selection of attractive future months or threshold tuning between them. To compare month reports, keep the same frozen code/protocol token.
- Full 5s+quiet sensitivity, fixed 28 matched trials, all retrospective 30bps reference samples at 5/15/30/60/120s; negative and null outcomes preserved. A retrospective reference is *not* actual physical onset and price quotes are not executable fills.
- Save the private ZIP, freeze token, plan and reports **locally**. Never upload them to public GitHub.


In [ ]:
# 1. Clean offline analyst VM and public code only.
import os,subprocess,sys,hashlib,json
from pathlib import Path
if any(Path(p).exists() for p in ('/content/drive/MyDrive','/content/gdrive/MyDrive','/gdrive/MyDrive')):
    raise RuntimeError('Drive mounted; delete runtime and use clean offline Colab.')
if any(os.environ.get(k) for k in ('GOOGLE_APPLICATION_CREDENTIALS','GOOGLE_OAUTH_ACCESS_TOKEN','GOOGLE_DRIVE_TOKEN')):
    raise RuntimeError('Credential env detected; fail closed')
repo=Path('/content/hypolab-v31-code')
if repo.exists():raise RuntimeError('Use a fresh Colab runtime, not an old checkout')
subprocess.run(['git','clone','--depth','1','https://github.com/DuStER1921/CryptoHypothesisLab.git',str(repo)],check=True,stdout=subprocess.DEVNULL)
commit=subprocess.check_output(['git','rev-parse','HEAD'],cwd=repo,text=True).strip()
sys.path.insert(0,str(repo/'src'))
from hypolab.premove_v31 import make_v31_plan,run_v31,freeze_token_sha256,FUTURE_DEVELOPMENT_MONTHS,DIAGNOSTIC_MONTH
from hypolab.micro_bundle import MAX_ARCHIVE_BYTES
from google.colab import files
print('V3.1 code commit:',commit)
print('Drive NOT mounted; original market sources inaccessible to this notebook.')


In [ ]:
# 2. Upload ONE already owner-approved MICRO ZIP. Verify independently before analysis.
uploaded=files.upload()
if len(uploaded)!=1:raise RuntimeError('Exactly one MICRO ZIP required')
name,raw=next(iter(uploaded.items()))
if not name.startswith('HYPOLAB_MICRO_DEV_') or not name.endswith('.zip') or not 0<len(raw)<=MAX_ARCHIVE_BYTES:
    raise RuntimeError('Wrong input type/size')
expected=input('Exact SHA-256 of this MICRO ZIP: ').strip().lower()
actual=hashlib.sha256(raw).hexdigest()
if actual!=expected:raise RuntimeError('MICRO ZIP SHA-256 mismatch; no analysis')
private=Path('/content/hypolab-v31-private');private.mkdir(mode=0o700,exist_ok=False)
packet=private/'input.zip'
with packet.open('xb') as f:
    os.chmod(packet,0o600)
    f.write(raw)
del uploaded,raw
plan=make_v31_plan(packet)
plan_fields=json.loads(plan)
month=plan_fields['month']
print('Verified private month:',month,'role:',plan_fields['kind'])
print('FROZEN V3.1 TOKEN SHA256:',plan_fields['freeze_token_sha256'])
if month!=DIAGNOSTIC_MONTH:
    if month not in FUTURE_DEVELOPMENT_MONTHS:raise RuntimeError('Unfrozen future month')
    owner_token=input('Paste previously saved FROZEN V3.1 TOKEN SHA256 from July: ').strip().lower()
    if owner_token!=plan_fields['freeze_token_sha256']:
        raise RuntimeError('Code/protocol drift; must not silently modify frozen test')


In [ ]:
# 3. Download and save exact frozen plan BEFORE outcomes.
plan_path=private/'premove-v31-frozen-plan.json'
with plan_path.open('xb') as f:f.write(plan)
print('Plan SHA-256:',hashlib.sha256(plan).hexdigest())
print('SAVE FREEZE TOKEN for every future-month run:',plan_fields['freeze_token_sha256'])
files.download(str(plan_path))


In [ ]:
# 4. Fixed V3.1 diagnostics and 28 descriptive matched trials.
report=run_v31(packet,plan)
print('V3.1 REPORT completed, month:',report['month'],'protocol token:',report['freeze_token_sha256'])
print('Retrospective reference count (NOT true onset):',report['reference']['count'])
print('Eligible primary 5s/12bps decisions:',report['risk_sensitivity']['all5_12bps']['eligible_risk_decisions'])
for name,scenario in report['risk_sensitivity'].items():
    print(name,'eligible=',scenario['eligible_risk_decisions'],
          'C2 surge prior 60s references=',scenario['pre_reference_directional_caught']['c2_flow_surge'])
for trial in report['trials']:
    if trial['threshold_bps']==15 and trial['horizon_seconds']==300:
        a=trial['train'];b=trial['later_development_diagnostic']
        print(trial['rule'],'early_pairs=',a['pairs'],'early_diff=',a['event_risk_difference'],
              'late_pairs=',b['pairs'],'late_diff=',b['event_risk_difference'])
print('Exploratory Development only; no OOS, execution edge or physical onset.')


In [ ]:
# 5. Diagnostic visualization only (NEVER select best threshold here).
import matplotlib.pyplot as plt
leads=[5,15,30,60,120]
quiet=[report['reference']['lead_stage_counts'][str(k)].get('QUIET_8BPS',0) for k in leads]
emerging=[report['reference']['lead_stage_counts'][str(k)].get('EMERGING_8_TO_20BPS',0) for k in leads]
active=[report['reference']['lead_stage_counts'][str(k)].get('ACTIVE_OVER_20BPS',0) for k in leads]
fig,ax=plt.subplots(figsize=(10,4))
ax.plot(leads,quiet,marker='o',label='Quiet <= 8bps')
ax.plot(leads,emerging,marker='o',label='Emerging >8..20bps')
ax.plot(leads,active,marker='o',label='Already moving >20bps')
ax.set(xlabel='Seconds before ex-post reference bin START',ylabel='Count of retrospective references',
       title='DESCRIPTIVE ONLY: past-known market stage vs future-selected references')
ax.grid(True,alpha=.2);ax.legend();plt.show()


In [ ]:
# 6. Download ALL failures and 5/15/30/60/120s samples in PRIVATE JSON.
report_path=private/'premove-v31-full-report.json'
with report_path.open('x',encoding='utf8') as f:
    json.dump(report,f,ensure_ascii=False,sort_keys=True,indent=2,allow_nan=False)
    f.write('\n')
files.download(str(report_path))
print('Keep original MICRO ZIP, plan, freeze token and report privately. Never public GitHub.')
